# `interrupt()`：动态暂停与恢复

> 适用版本：本项目锁定的 **LangGraph 1.1.2** 与 **langgraph-checkpoint 4.0.1**。本 Notebook 只运行本地确定性代码，不调用大模型、网络或外部服务。

`interrupt()` 用于在节点内部动态暂停图，把问题或审批上下文交给图外的人或系统；稍后再用 `Command(resume=...)` 把回答送回同一个执行线程。它和 `interrupt_before` / `interrupt_after` 这类静态断点不同：`interrupt()` 可以出现在节点代码的任意位置，也可以根据业务条件决定是否触发。

本章将直接验证：

1. 第一次调用怎样从完整返回值的 `__interrupt__` 字段拿到中断事件；
2. 为什么可恢复中断必须配置 checkpointer；
3. 为什么恢复时必须复用同一个 `thread_id`；
4. `Command(resume=...)` 的值怎样成为 `interrupt()` 的返回值；
5. 恢复时为什么不是从 Python 源码的下一行继续，而是从节点开头重跑；
6. 节点重跑、失败重试和 replay 对外部副作用提出什么幂等性要求。

```mermaid
flowchart TD
    A[首次 invoke: 初始 State + thread_id] --> B[checkpointer 读取该线程]
    B --> C[approval 节点从开头执行]
    C --> D[interrupt: 提交审批请求]
    D --> E[保存 State、待执行任务与中断信息]
    E --> F[调用者收到 __interrupt__]
    F --> G[图外获得人工回答]
    G --> H[同一 thread_id + Command resume]
    H --> I[恢复 checkpoint，approval 节点从开头重跑]
    I --> J[interrupt 返回 resume 值]
    J --> K[节点提交 State 更新]
    K --> L[下游 apply_decision 节点]
```

In [1]:
from importlib.metadata import version
from pprint import pprint
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

print("LangGraph version:", version("langgraph"))
print("Checkpoint version:", version("langgraph-checkpoint"))

LangGraph version: 1.1.2
Checkpoint version: 4.0.1


## 1. 先理解暂停语义

节点第一次执行到 `interrupt(payload)` 时，`interrupt()` **不会像普通函数那样返回**。它会抛出 LangGraph 内部用于控制流程的特殊异常；图运行时捕获该异常，记录中断信息，并把 `payload` 暴露给调用者。对默认的 `invoke(..., version="v1")`，中断事件位于完整返回字典的 `__interrupt__` 字段。

这里的“暂停”不是让当前 `invoke()` 一直阻塞：首次 `invoke()` 会携带中断事件返回，线程的可恢复状态留在 checkpointer 中，图外代码可以几秒、几小时甚至几天后再恢复。

checkpointer 保存的是 LangGraph State、调度任务与中断元数据，**不会序列化 Python 调用栈或保存源码行指针**。因此恢复时，包含 `interrupt()` 的整个节点会从函数开头重新执行；运行到同一个 `interrupt()` 后，运行时找到已提供的 resume 值，这一次 `interrupt()` 才把该值返回给节点代码。

> 不要用宽泛的 `try/except` 包住 `interrupt()`：暂停依赖内部异常传播。若节点有多个 `interrupt()`，还必须保持调用顺序稳定，因为恢复值按调用顺序匹配。

## 2. 构建一个本地审批图

示例模拟“发布课程草稿前请求人工审批”，不执行真实发布操作。图包含两个节点：

- `request_approval`：记录一次节点入口，然后调用 `interrupt()`；恢复后把人工回答写入 `decision`；
- `apply_decision`：根据回答生成本地结果。

三个 Python 列表故意放在 State 外，用作本进程的观察日志。它们不受 checkpointer 管理，正好帮助我们看见节点函数实际执行了多少次。

In [2]:
class ApprovalState(TypedDict, total=False):
    operation_id: str
    request: str
    decision: dict[str, object]
    outcome: str


# 这些列表位于 LangGraph State 之外，只用于观察本进程中的执行次数。
node_entry_log: list[dict[str, object]] = []
interrupt_return_log: list[dict[str, object]] = []
action_run_log: list[dict[str, object]] = []


def request_approval(state: ApprovalState) -> dict[str, object]:
    # 这段代码位于 interrupt() 之前，恢复时会再次执行。
    node_entry_log.append(
        {
            "entry_number": len(node_entry_log) + 1,
            "operation_id": state["operation_id"],
        }
    )

    human_response = interrupt(
        {
            "kind": "approval_request",
            "operation_id": state["operation_id"],
            "question": f"是否批准：{state['request']}？",
            "allowed_actions": ["approve", "reject"],
        }
    )

    # 首次执行不会到达这里；恢复时 interrupt() 才会返回 resume 值。
    interrupt_return_log.append(human_response)
    return {"decision": human_response}


def apply_decision(state: ApprovalState) -> dict[str, str]:
    # 这里只模拟下游动作，不调用真实外部服务。
    action_run_log.append(
        {
            "operation_id": state["operation_id"],
            "decision": state["decision"],
        }
    )
    action = state["decision"]["action"]
    outcome = (
        f"已批准：{state['request']}"
        if action == "approve"
        else f"已拒绝：{state['request']}"
    )
    return {"outcome": outcome}


builder = StateGraph(ApprovalState)
builder.add_node("request_approval", request_approval)
builder.add_node("apply_decision", apply_decision)
builder.add_edge(START, "request_approval")
builder.add_edge("request_approval", "apply_decision")
builder.add_edge("apply_decision", END)

memory_saver = InMemorySaver()
approval_graph = builder.compile(checkpointer=memory_saver)

print("审批图已使用 InMemorySaver 编译。")

审批图已使用 InMemorySaver 编译。


## 3. 配置稳定的 `thread_id`

`thread_id` 是 checkpointer 查找暂停线程的指针，放在运行配置的 `configurable` 中，不属于业务 State。首次调用和恢复调用必须使用同一个值；最稳妥的写法是复用同一个 `thread_config` 对象。

本 Notebook 使用固定教学 ID，并在运行前只删除这个 ID 的内存记录，使“从头运行全部单元格”具有稳定语义。真实应用不应在等待恢复时删除线程。

In [3]:
THREAD_ID = "interrupt-approval-demo"
thread_config = {"configurable": {"thread_id": THREAD_ID}}

# 仅清理本 Notebook 的确定性线程和本地观察日志。
memory_saver.delete_thread(THREAD_ID)
node_entry_log.clear()
interrupt_return_log.clear()
action_run_log.clear()

print("本教程使用的完整运行配置：")
pprint(thread_config, sort_dicts=False)

本教程使用的完整运行配置：
{'configurable': {'thread_id': 'interrupt-approval-demo'}}


## 4. 首次调用：从 `__interrupt__` 拿到中断事件

首次调用传入普通 State 输入。`request_approval` 执行到 `interrupt()` 后暂停，因此 `apply_decision` 尚未运行，最终业务结果也尚未生成。

下面先完整输出 `invoke()` 的原始返回值、完整 `__interrupt__` 字段和节点入口日志，再给出便于阅读的字段摘要。

In [4]:
first_result = approval_graph.invoke(
    {
        "operation_id": "publish-demo-001",
        "request": "发布 interrupt 教程草稿",
    },
    thread_config,
    durability="sync",
)

print("首次 invoke 的完整原始返回值：")
pprint(first_result, sort_dicts=False)
print("\n完整 __interrupt__ 字段：")
pprint(first_result["__interrupt__"], sort_dicts=False)
print("\n暂停后的完整节点入口日志：")
pprint(node_entry_log, sort_dicts=False)
print("\n暂停后的完整 interrupt 返回日志：")
pprint(interrupt_return_log, sort_dicts=False)
print("\n暂停后的完整下游动作日志：")
pprint(action_run_log, sort_dicts=False)

first_interrupt = first_result["__interrupt__"][0]
print("\n友好摘要：")
print("Interrupt.value：", first_interrupt.value)
print("Interrupt.id：", first_interrupt.id)
print("当前节点入口次数：", len(node_entry_log))

首次 invoke 的完整原始返回值：
{'operation_id': 'publish-demo-001',
 'request': '发布 interrupt 教程草稿',
 '__interrupt__': [Interrupt(value={'kind': 'approval_request',
                                    'operation_id': 'publish-demo-001',
                                    'question': '是否批准：发布 interrupt 教程草稿？',
                                    'allowed_actions': ['approve', 'reject']},
                             id='4c873a93e5c6318e83162f30c7876436')]}

完整 __interrupt__ 字段：
[Interrupt(value={'kind': 'approval_request',
                  'operation_id': 'publish-demo-001',
                  'question': '是否批准：发布 interrupt 教程草稿？',
                  'allowed_actions': ['approve', 'reject']},
           id='4c873a93e5c6318e83162f30c7876436')]

暂停后的完整节点入口日志：
[{'entry_number': 1, 'operation_id': 'publish-demo-001'}]

暂停后的完整 interrupt 返回日志：
[]

暂停后的完整下游动作日志：
[]

友好摘要：
Interrupt.value： {'kind': 'approval_request', 'operation_id': 'publish-demo-001', 'question': '是否批准：发布 interrupt 教程草稿？', 'allowed_acti

### 结果解读

- 完整结果仍含首次输入的 `operation_id` 与 `request`，并新增 `__interrupt__`；此时没有 `decision` 和 `outcome`。
- `__interrupt__` 中的 `Interrupt.value` 就是节点传给 `interrupt()` 的完整审批请求；`Interrupt.id` 标识这一次具体中断，它不是业务 `thread_id`。
- `node_entry_log` 已有一条记录，证明节点开头执行过一次。
- `interrupt_return_log` 仍为空，证明首次执行的 `interrupt()` 没有返回到下一行。
- `action_run_log` 仍为空，证明下游节点尚未执行。

## 5. 查看暂停时保存的 checkpoint

下面先完整输出 `StateSnapshot`，随后再完整输出几个关键字段。重点观察：

- `values` 只有暂停前已经提交的 State；
- `next=('request_approval',)` 表示该节点仍是待完成任务；
- `tasks` / `interrupts` 保存了中断信息；
- `config.configurable` 同时带有 `thread_id`、`checkpoint_ns` 与自动生成的 `checkpoint_id`。

In [5]:
paused_snapshot = approval_graph.get_state(thread_config)

print("暂停后的完整 StateSnapshot：")
pprint(paused_snapshot, sort_dicts=False)
print("\n完整 values 字段：")
pprint(paused_snapshot.values, sort_dicts=False)
print("\n完整 next 字段：")
pprint(paused_snapshot.next, sort_dicts=False)
print("\n完整 tasks 字段：")
pprint(paused_snapshot.tasks, sort_dicts=False)
print("\n完整 interrupts 字段：")
pprint(paused_snapshot.interrupts, sort_dicts=False)

print("\n友好摘要：")
print("待执行节点：", paused_snapshot.next)
print("checkpoint_id：", paused_snapshot.config["configurable"]["checkpoint_id"])

暂停后的完整 StateSnapshot：
StateSnapshot(values={'operation_id': 'publish-demo-001', 'request': '发布 interrupt 教程草稿'}, next=('request_approval',), config={'configurable': {'thread_id': 'interrupt-approval-demo', 'checkpoint_ns': '', 'checkpoint_id': '1f19d41d-7f7b-6350-8000-953c1a552539'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-21T09:22:34.696574+00:00', parent_config={'configurable': {'thread_id': 'interrupt-approval-demo', 'checkpoint_ns': '', 'checkpoint_id': '1f19d41d-7f79-6af0-bfff-a77af60a76a9'}}, tasks=(PregelTask(id='acb32b0e-de89-a440-298d-e0a97a0196e7', name='request_approval', path=('__pregel_pull', 'request_approval'), error=None, interrupts=(Interrupt(value={'kind': 'approval_request', 'operation_id': 'publish-demo-001', 'question': '是否批准：发布 interrupt 教程草稿？', 'allowed_actions': ['approve', 'reject']}, id='4c873a93e5c6318e83162f30c7876436'),), state=None, result=None),), interrupts=(Interrupt(value={'kind': 'approval_request', 'operation_id': 

### 为什么需要 checkpointer

恢复不是简单地再次调用 Python 函数。运行时需要先按 `thread_id` 找到暂停 checkpoint，恢复 State、待执行任务、interrupt 元数据以及已提供的 resume 值，再重新调度节点。没有 checkpointer，就没有可供后续调用恢复的线程时间线。

本例显式使用 `durability="sync"`，使 checkpoint 在后续步骤开始前同步写入 saver，方便紧接着检查；它不改变节点重跑语义，也不会给外部系统提供 exactly-once 保证。

## 6. 使用 `Command(resume=...)` 恢复

恢复调用有两个关键点：

1. 输入不再是新的 State 字典，而是 `Command(resume=resume_payload)`；
2. 配置继续使用同一个 `thread_config`，因此 `thread_id` 完全相同。

恢复后，`request_approval` 从函数开头重跑。到达同一个 `interrupt()` 时，这次调用返回 `resume_payload`；节点随后把它写入 `decision`，图再运行下游节点。

In [6]:
resume_payload = {
    "action": "approve",
    "reviewer": "教学用户",
    "comment": "已核对，仅执行本地模拟",
}
resume_command = Command(resume=resume_payload)

print("恢复时输入的完整 Command：")
pprint(resume_command, sort_dicts=False)

resumed_result = approval_graph.invoke(
    resume_command,
    thread_config,  # 与首次调用复用完全相同的 thread_id。
    durability="sync",
)

print("\n恢复后的完整 invoke 返回值：")
pprint(resumed_result, sort_dicts=False)
print("\n恢复后的完整节点入口日志：")
pprint(node_entry_log, sort_dicts=False)
print("\n恢复后的完整 interrupt 返回日志：")
pprint(interrupt_return_log, sort_dicts=False)
print("\n恢复后的完整下游动作日志：")
pprint(action_run_log, sort_dicts=False)

print("\n友好摘要：")
print("interrupt() 返回值是否等于 resume_payload：", interrupt_return_log[0] == resume_payload)
print("节点入口总次数：", len(node_entry_log))
print("下游节点执行次数：", len(action_run_log))

恢复时输入的完整 Command：
Command(resume={'action': 'approve', 'reviewer': '教学用户', 'comment': '已核对，仅执行本地模拟'})

恢复后的完整 invoke 返回值：
{'operation_id': 'publish-demo-001',
 'request': '发布 interrupt 教程草稿',
 'decision': {'action': 'approve',
              'reviewer': '教学用户',
              'comment': '已核对，仅执行本地模拟'},
 'outcome': '已批准：发布 interrupt 教程草稿'}

恢复后的完整节点入口日志：
[{'entry_number': 1, 'operation_id': 'publish-demo-001'},
 {'entry_number': 2, 'operation_id': 'publish-demo-001'}]

恢复后的完整 interrupt 返回日志：
[{'action': 'approve', 'reviewer': '教学用户', 'comment': '已核对，仅执行本地模拟'}]

恢复后的完整下游动作日志：
[{'operation_id': 'publish-demo-001',
  'decision': {'action': 'approve',
               'reviewer': '教学用户',
               'comment': '已核对，仅执行本地模拟'}}]

友好摘要：
interrupt() 返回值是否等于 resume_payload： True
节点入口总次数： 2
下游节点执行次数： 1


### 结果解读

| 观察 | 原始输出中的证据 | 机制 |
| --- | --- | --- |
| resume 值成为 `interrupt()` 返回值 | `interrupt_return_log[0]` 与 `resumed_result['decision']` 都是完整 `resume_payload` | 恢复值在节点重跑到同一 interrupt 时被取回 |
| 节点从开头重跑 | `node_entry_log` 有两条记录 | 首次暂停前执行一次，恢复时再执行一次 |
| interrupt 后代码首次未执行 | `interrupt_return_log` 最终只有一条记录 | 首次 interrupt 抛出内部控制流异常；仅恢复执行会越过该点 |
| 下游节点在正常恢复路径执行一次 | `action_run_log` 只有一条记录 | 首次暂停时尚未调度，恢复并提交 decision 后才调度 |

若恢复时换成另一个 `thread_id`，checkpointer 会定位另一条（通常为空的）时间线，而不是这个暂停点。具体表现可能是从空 State 开始、再次中断或因缺字段报错，但都不属于原线程恢复。

## 7. 边界验证：没有 checkpointer 会怎样

在 LangGraph 1.1.2 中，不带 checkpointer 的图第一次仍能把 `__interrupt__` 暴露给调用者；但它没有保存可恢复 checkpoint，所以之后的 `Command(resume=...)` 会失败。下面先输出首次调用完整结果，再输出恢复时的完整异常对象。

这也是“`interrupt()` 必须配置 checkpointer”的精确含义：**可见一次中断不等于具备可恢复的中断工作流**。

In [7]:
class NoCheckpointState(TypedDict, total=False):
    prompt: str
    answer: str


def pause_without_checkpoint(
    state: NoCheckpointState,
) -> dict[str, str]:
    answer = interrupt(state["prompt"])
    return {"answer": answer}


no_checkpoint_builder = StateGraph(NoCheckpointState)
no_checkpoint_builder.add_node(
    "pause_without_checkpoint",
    pause_without_checkpoint,
)
no_checkpoint_builder.add_edge(START, "pause_without_checkpoint")
no_checkpoint_builder.add_edge("pause_without_checkpoint", END)
no_checkpoint_graph = no_checkpoint_builder.compile()
no_checkpoint_config = {
    "configurable": {"thread_id": "no-checkpointer-demo"}
}

no_checkpoint_first_result = no_checkpoint_graph.invoke(
    {"prompt": "请输入任意回答"},
    no_checkpoint_config,
)
print("没有 checkpointer 时首次调用的完整返回值：")
pprint(no_checkpoint_first_result, sort_dicts=False)

try:
    no_checkpoint_graph.invoke(
        Command(resume="这是无法恢复的回答"),
        no_checkpoint_config,
    )
except RuntimeError as exc:
    resume_without_checkpoint_error = exc
else:
    resume_without_checkpoint_error = None

print("\n没有 checkpointer 时恢复得到的完整异常对象：")
pprint(resume_without_checkpoint_error)
print("\n友好摘要：")
print(
    "异常类型：",
    type(resume_without_checkpoint_error).__name__,
)
print("异常消息：", str(resume_without_checkpoint_error))

没有 checkpointer 时首次调用的完整返回值：
{'prompt': '请输入任意回答',
 '__interrupt__': [Interrupt(value='请输入任意回答',
                             id='bda312aea1588bfd813c9c0dd6928959')]}

没有 checkpointer 时恢复得到的完整异常对象：
RuntimeError('Cannot use Command(resume=...) without checkpointer')

友好摘要：
异常类型： RuntimeError
异常消息： Cannot use Command(resume=...) without checkpointer


## 8. 外部副作用与幂等性边界

`node_entry_log` 已经直接证明：`interrupt()` 之前的代码在“首次暂停 + 一次恢复”中执行了两次。若那一段不是追加教学日志，而是扣款、发邮件、创建工单或写外部数据库，就可能产生重复副作用。

| 代码位置 | 暂停/恢复时的行为 | 工程要求 |
| --- | --- | --- |
| `interrupt()` 之前 | 恢复时必然重跑 | 尽量只做纯计算；外部写操作必须幂等 |
| `interrupt()` 之后、仍在同一节点 | 正常恢复只执行一次，但节点失败重试时仍可能重复 | 仍需幂等，不要假设 exactly-once |
| 独立下游节点 | 能避免因“回到 interrupt”而重复，但 retry、replay、time travel 或 worker 崩溃仍可重跑 | 继续使用业务幂等键、唯一约束或 outbox |

推荐让 `operation_id` 之类的稳定业务 ID 成为幂等键，并让真正持有副作用的系统用唯一约束、幂等 API、去重表或 transactional outbox 落实“重复请求只生效一次”。checkpointer 只管理 LangGraph 的执行状态，不会和外部系统自动组成原子事务；`durability="sync"` 也不能改变这一点。

## 9. `InMemorySaver` 的生命周期与适用边界

本章按教学要求使用 `InMemorySaver`。要成功恢复，除了 `thread_id` 相同，还必须继续使用持有原 checkpoint 的同一个 saver 对象：

- 适合 Notebook、单元测试和单进程调试；
- 进程退出、内核重启或新建另一个 `InMemorySaver()` 后，原暂停线程不会保留；
- 多进程 worker 之间也不会共享这份内存；
- 生产中的长时间人工审批应改用数据库支持的持久化 checkpointer，并配置备份、并发控制与保留策略。

因此本例中的“稍后恢复”只在当前 Python 进程和当前 `memory_saver` 实例存活期间成立。

## 10. 最终状态与核对摘要

下面先完整输出恢复完成后的最新 `StateSnapshot` 和三份观察日志，再给出基于这些原始对象的核对摘要。

In [8]:
final_snapshot = approval_graph.get_state(thread_config)

print("恢复完成后的完整 StateSnapshot：")
pprint(final_snapshot, sort_dicts=False)
print("\n最终完整节点入口日志：")
pprint(node_entry_log, sort_dicts=False)
print("\n最终完整 interrupt 返回日志：")
pprint(interrupt_return_log, sort_dicts=False)
print("\n最终完整下游动作日志：")
pprint(action_run_log, sort_dicts=False)

verification_summary = {
    "首次调用暴露中断": "__interrupt__" in first_result,
    "暂停时待执行节点": paused_snapshot.next,
    "resume 值成为 interrupt 返回值": interrupt_return_log[0]
    == resume_payload,
    "包含 interrupt 的节点入口次数": len(node_entry_log),
    "下游节点执行次数": len(action_run_log),
    "最终 next": final_snapshot.next,
    "无 checkpointer 恢复异常": repr(
        resume_without_checkpoint_error
    ),
}
print("\n基于上述完整原始对象的友好核对摘要：")
pprint(verification_summary, sort_dicts=False)

恢复完成后的完整 StateSnapshot：
StateSnapshot(values={'operation_id': 'publish-demo-001', 'request': '发布 interrupt 教程草稿', 'decision': {'action': 'approve', 'reviewer': '教学用户', 'comment': '已核对，仅执行本地模拟'}, 'outcome': '已批准：发布 interrupt 教程草稿'}, next=(), config={'configurable': {'thread_id': 'interrupt-approval-demo', 'checkpoint_ns': '', 'checkpoint_id': '1f19d41d-7f9f-65c0-8002-561eb308fe60'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-21T09:22:34.711385+00:00', parent_config={'configurable': {'thread_id': 'interrupt-approval-demo', 'checkpoint_ns': '', 'checkpoint_id': '1f19d41d-7f9c-6dc0-8001-de12a426fced'}}, tasks=(), interrupts=())

最终完整节点入口日志：
[{'entry_number': 1, 'operation_id': 'publish-demo-001'},
 {'entry_number': 2, 'operation_id': 'publish-demo-001'}]

最终完整 interrupt 返回日志：
[{'action': 'approve', 'reviewer': '教学用户', 'comment': '已核对，仅执行本地模拟'}]

最终完整下游动作日志：
[{'operation_id': 'publish-demo-001',
  'decision': {'action': 'approve',
               'reviewer': 

## 11. 总结

- 首次执行到 `interrupt(payload)` 时，节点暂停，调用者从 `result['__interrupt__']` 取得完整 `Interrupt`；
- checkpointer 保存可恢复 State 和调度信息，`thread_id` 定位线程时间线；
- 恢复时传入 `Command(resume=value)` 并复用同一个 `thread_id`；
- 节点从函数开头重跑，到同一 `interrupt()` 后，`value` 成为该调用的返回值；
- `interrupt()` 前的逻辑必然再次执行，所有外部副作用都要按至少一次执行语义设计幂等；
- `InMemorySaver` 只在当前进程和 saver 实例内保留 checkpoint，生产审批流程应使用持久化后端。

### 官方资料

- [LangGraph：Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [LangGraph API Reference：interrupt](https://reference.langchain.com/python/langgraph/types/interrupt)
- [LangGraph API Reference：Command](https://reference.langchain.com/python/langgraph/types/Command)
- [LangGraph：Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)